In [1]:
import pandas as pd 

df = pd.read_csv("cleaned_data.csv")

In [2]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805549 entries, 0 to 805548
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      805549 non-null  int64  
 1   StockCode    805549 non-null  object 
 2   Description  805549 non-null  object 
 3   Quantity     805549 non-null  int64  
 4   InvoiceDate  805549 non-null  object 
 5   Price        805549 non-null  float64
 6   Customer ID  805549 non-null  int64  
 7   Country      805549 non-null  object 
 8   TotalPrice   805549 non-null  float64
dtypes: float64(2), int64(3), object(4)
memory usage: 55.3+ MB


In [3]:
# Converting InvoiceDate to actual datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [4]:
import datetime as dt

reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)
print(reference_date)

2011-12-10 12:50:00


In [5]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,   # Recency
    'Invoice': 'nunique',                                        # Frequency
    'TotalPrice': 'sum'                                          # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [6]:
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40
5,12351,375,1,300.93
6,12352,36,10,2849.84
7,12353,204,2,406.76
8,12354,232,1,1079.40
9,12355,214,2,947.61


In [7]:
# Recency: LOWER days = BETTER, so we score in REVERSE (labels 5,4,3,2,1)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1])

# Frequency: HIGHER orders = BETTER, so normal order (labels 1,2,3,4,5)
# Using rank(method='first') to avoid errors from too many repeated/tied values
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# Monetary: HIGHER spend = BETTER, so normal order (labels 1,2,3,4,5)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1,2,3,4,5])

rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,326,12,77556.46,2,5,5
1,12347,2,8,5633.32,5,4,5
2,12348,75,5,2019.40,3,4,4
3,12349,19,4,4428.69,5,3,5
4,12350,310,1,334.40,2,1,2
5,12351,375,1,300.93,2,1,2
6,12352,36,10,2849.84,4,5,4
7,12353,204,2,406.76,2,2,2
8,12354,232,1,1079.40,2,1,3
9,12355,214,2,947.61,2,2,3


In [8]:
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment
0,12346,326,12,77556.46,2,5,5,255
1,12347,2,8,5633.32,5,4,5,545
2,12348,75,5,2019.40,3,4,4,344
3,12349,19,4,4428.69,5,3,5,535
4,12350,310,1,334.40,2,1,2,212
5,12351,375,1,300.93,2,1,2,212
6,12352,36,10,2849.84,4,5,4,454
7,12353,204,2,406.76,2,2,2,222
8,12354,232,1,1079.40,2,1,3,213
9,12355,214,2,947.61,2,2,3,223


In [9]:
def label_segment(row):
    r, f, m = int(row['R_Score']), int(row['F_Score']), int(row['M_Score'])
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 4 and f <= 2:
        return 'New/Promising'
    elif r <= 2 and f >= 4 and m >= 4:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost/Hibernating'
    else:
        return 'Regular/Needs Attention'

rfm['Customer_Segment'] = rfm.apply(label_segment, axis=1)
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,Customer_Segment
0,12346,326,12,77556.46,2,5,5,255,At Risk
1,12347,2,8,5633.32,5,4,5,545,Champions
2,12348,75,5,2019.40,3,4,4,344,Regular/Needs Attention
3,12349,19,4,4428.69,5,3,5,535,Regular/Needs Attention
4,12350,310,1,334.40,2,1,2,212,Lost/Hibernating
5,12351,375,1,300.93,2,1,2,212,Lost/Hibernating
6,12352,36,10,2849.84,4,5,4,454,Champions
7,12353,204,2,406.76,2,2,2,222,Lost/Hibernating
8,12354,232,1,1079.40,2,1,3,213,Regular/Needs Attention
9,12355,214,2,947.61,2,2,3,223,Regular/Needs Attention


In [10]:
rfm['Customer_Segment'].value_counts()

Customer_Segment
Regular/Needs Attention    2633
Champions                  1300
Lost/Hibernating           1275
New/Promising               443
At Risk                     227
Name: count, dtype: int64

In [11]:
rfm['Customer_Segment'].isnull().sum()
rfm[rfm['R_Score'].isnull() | rfm['F_Score'].isnull() | rfm['M_Score'].isnull()]

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,Customer_Segment


## Revenue per segment 

In [13]:
segment_summary = rfm.groupby('Customer_Segment').agg({
    'CustomerID': 'count',
    'Monetary': ['sum', 'mean']
}).round(2)

segment_summary.columns = ['Customer_Count', 'Total_Revenue', 'Avg_Revenue_Per_Customer']
segment_summary['Pct_of_Customers'] = (segment_summary['Customer_Count'] / segment_summary['Customer_Count'].sum() * 100).round(2)
segment_summary['Pct_of_Revenue'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100).round(2)
segment_summary = segment_summary.sort_values('Total_Revenue', ascending=False)
segment_summary

,Customer_Count,Total_Revenue,Avg_Revenue_Per_Customer,Pct_of_Customers,Pct_of_Revenue
Customer_Segment,,,,,
Champions,1300,12128115.56,9329.32,22.12,68.35
Regular/Needs Attention,2633,3874165.07,1471.39,44.79,21.83
At Risk,227,1018866.68,4488.40,3.86,5.74
New/Promising,443,394638.61,890.83,7.54,2.22
Lost/Hibernating,1275,327643.25,256.98,21.69,1.85


In [14]:
# Save RFM dataset to the data folder
rfm.to_csv("rfm_results.csv", index=False)

print("RFM dataset saved successfully!")

RFM dataset saved successfully!
